In [ ]:
import os
import sys
import subprocess
import json
import requests
import time

from google.colab import drive, userdata
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

REPO_DIR = "/content/teachmewrite"
REPO_URL = "https://github.com/edmedpublic-hub/teachmewrite.git"

if os.path.exists(REPO_DIR):
    print("Repo exists. Pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "anthropic"], check=True)

try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    print("Secrets loaded successfully.")
except Exception as e:
    print(f"ERROR loading secrets: {e}")
    raise

LIVE_URL = "https://teachmewrite-ai-195673983724.asia-south1.run.app"

assert os.environ.get("ANTHROPIC_API_KEY") is not None, "ANTHROPIC_API_KEY is missing!"
print(f"Target Live Service: {LIVE_URL}")
print("✅ Cell 1a complete. Environment is ready.")

Mounted at /content/drive
Cloning repo...
Installing dependencies...
Secrets loaded successfully.
Target Live Service: https://teachmewrite-ai-195673983724.asia-south1.run.app
✅ Cell 1a complete. Environment is ready.


In [ ]:
class AgentTracer:
    def __init__(self):
        self.start_time = time.time()
        self.events = []

    def log(self, event_type: str, **details):
        elapsed = time.time() - self.start_time
        self.events.append({
            "elapsed_s": round(elapsed, 3),
            "type": event_type,
            **details
        })

    def print_timeline(self):
        print("\n" + "=" * 60)
        print("AGENT TRACE TIMELINE")
        print("=" * 60)
        for ev in self.events:
            t = f"[+{ev['elapsed_s']:.2f}s]"
            etype = ev["type"]

            if etype == "user_message":
                print(f"{t} USER: {ev['content'][:80]}")
            elif etype == "llm_call":
                print(f"{t} LLM CALL (iteration {ev['iteration']})")
            elif etype == "llm_response":
                print(f"{t}   -> LLM responded with stop_reason='{ev['stop_reason']}' (latency: {ev['latency_ms']}ms)")
            elif etype == "tool_call":
                print(f"{t} TOOL CALL: {ev['tool_name']}")
            elif etype == "tool_result":
                status = "ERROR" if "error" in ev.get("result", {}) else "OK"
                print(f"{t}   -> Tool result: {status} (latency: {ev['latency_ms']}ms)")
            elif etype == "final_answer":
                print(f"{t} FINAL ANSWER ({ev['length']} chars)")
            elif etype == "error":
                print(f"{t} ERROR: {ev['message']}")
            else:
                print(f"{t} {etype}: {ev}")
        print("=" * 60)
        print(f"Total elapsed: {time.time() - self.start_time:.2f}s")
        print("=" * 60)

print("✅ Cell 1b complete. AgentTracer is ready.")

✅ Cell 1b complete. AgentTracer is ready.


In [ ]:
from anthropic import Anthropic
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# Minimal tool set (same as Day 92)
tools = [
    {
        "name": "get_writing_feedback",
        "description": "Get structured writing feedback. Use when the student wants to understand mistakes.",
        "input_schema": {
            "type": "object",
            "properties": {
                "student_text": {"type": "string"},
                "level": {"type": "integer"}
            },
            "required": ["student_text", "level"]
        }
    },
    {
        "name": "get_rewrite_suggestion",
        "description": "Get ONLY the corrected version of the text. Use when student says 'just fix it'.",
        "input_schema": {
            "type": "object",
            "properties": {
                "student_text": {"type": "string"},
                "level": {"type": "integer"}
            },
            "required": ["student_text", "level"]
        }
    }
]

def get_writing_feedback(student_text: str, level: int) -> dict:
    r = requests.post(f"{LIVE_URL}/feedback",
                      json={"student_text": student_text, "level": level}, timeout=30)
    r.raise_for_status()
    return r.json()

def get_rewrite_suggestion(student_text: str, level: int) -> dict:
    r = requests.post(f"{LIVE_URL}/feedback",
                      json={"student_text": student_text, "level": level}, timeout=30)
    r.raise_for_status()
    return {"rewrite_suggestion": r.json().get("rewrite_suggestion", "")}

TOOL_FUNCTIONS = {
    "get_writing_feedback": get_writing_feedback,
    "get_rewrite_suggestion": get_rewrite_suggestion,
}


class TracedAgent:
    def __init__(self):
        self.tracer = AgentTracer()
        self.system_prompt = """You are an English writing tutor for Pakistani students.
You have two tools:
1. get_writing_feedback — for "help me understand"
2. get_rewrite_suggestion — for "just fix it"

Decide which tool fits the user's intent. Be brief and encouraging.
"""

    def chat(self, user_message: str, level: int = 5, max_iterations: int = 5) -> dict:
        self.tracer.log("user_message", content=user_message)
        messages = [{"role": "user", "content": user_message}]

        for iteration in range(max_iterations):
            # --- LLM CALL ---
            self.tracer.log("llm_call", iteration=iteration + 1)
            t0 = time.time()
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=1024,
                system=self.system_prompt,
                tools=tools,
                messages=messages
            )
            llm_latency = int((time.time() - t0) * 1000)
            self.tracer.log("llm_response",
                            stop_reason=response.stop_reason,
                            latency_ms=llm_latency)

            if response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                tool_results = []

                for block in response.content:
                    if block.type == "tool_use":
                        tool_name = block.name
                        self.tracer.log("tool_call", tool_name=tool_name)

                        t0 = time.time()
                        try:
                            result = TOOL_FUNCTIONS[tool_name](**block.input)
                        except Exception as e:
                            result = {"error": str(e)}
                            self.tracer.log("error", message=str(e))
                        tool_latency = int((time.time() - t0) * 1000)

                        self.tracer.log("tool_result",
                                        tool_name=tool_name,
                                        latency_ms=tool_latency,
                                        result=result)

                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result)
                        })

                messages.append({"role": "user", "content": tool_results})

            elif response.stop_reason == "end_turn":
                final_text = "".join(b.text for b in response.content if hasattr(b, "text"))
                self.tracer.log("final_answer", length=len(final_text))
                return {"response": final_text, "tracer": self.tracer}

            else:
                self.tracer.log("error", message=f"Unexpected stop_reason: {response.stop_reason}")
                return {"response": "Unexpected error", "tracer": self.tracer}

        self.tracer.log("error", message="Max iterations reached")
        return {"response": "Max iterations reached.", "tracer": self.tracer}

print("✅ Cell 1c complete. TracedAgent is ready.")

✅ Cell 1c complete. TracedAgent is ready.


In [ ]:
student_text = "I am agree with my father that education is more better than money."
query = f"{student_text} — Just fix it for me."

print("=" * 60)
print("RUNNING TRACED AGENT")
print("=" * 60)
print(f"Query: {query}\n")

agent = TracedAgent()
result = agent.chat(query, level=5)

print("\n--- AGENT RESPONSE ---")
print(result["response"])

# Print the trace timeline
result["tracer"].print_timeline()

# Verification
assert len(result["tracer"].events) >= 4, "Trace should have multiple events"
assert any(e["type"] == "final_answer" for e in result["tracer"].events), "Should have a final answer event"
assert any(e["type"] == "tool_call" for e in result["tracer"].events), "Should have a tool call event"
print("✅ Cell 1d complete. Trace captured and rendered.")

RUNNING TRACED AGENT
Query: I am agree with my father that education is more better than money. — Just fix it for me.


--- AGENT RESPONSE ---
Let me fix this for you right away:

**Corrected version:**
"I **agree** with my father that education is **better** than money."

**Quick notes:**
- Remove "am" before "agree" (use "I agree" not "I am agree")
- Change "more better" to just "better" (don't use both "more" and "better" together)

Well done on expressing your opinion! 👍

AGENT TRACE TIMELINE
[+0.00s] USER: I am agree with my father that education is more better than money. — Just fix i
[+0.00s] LLM CALL (iteration 1)
[+1.38s]   -> LLM responded with stop_reason='tool_use' (latency: 1375ms)
[+1.38s] TOOL CALL: get_rewrite_suggestion
[+31.42s] ERROR: HTTPSConnectionPool(host='teachmewrite-ai-195673983724.asia-south1.run.app', port=443): Read timed out. (read timeout=30)
[+31.42s]   -> Tool result: ERROR (latency: 30039ms)
[+31.42s] LLM CALL (iteration 2)
[+32.93s]   -> LLM responded

In [ ]:
# Warm-instance test: same query, should be much faster
print("=" * 60)
print("WARM INSTANCE TEST")
print("=" * 60)

t0 = time.time()
warm_result = agent.chat(query, level=5)
warm_elapsed = time.time() - t0

print(f"\nWarm run elapsed: {warm_elapsed:.2f}s")
warm_result["tracer"].print_timeline()

if warm_elapsed < 10:
    print("✅ WARM RUN FAST — confirms cold-start was the cause.")
else:
    print("⚠️ STILL SLOW — timeout may be a deeper issue.")

WARM INSTANCE TEST

Warm run elapsed: 7.26s

AGENT TRACE TIMELINE
[+0.00s] USER: I am agree with my father that education is more better than money. — Just fix i
[+0.00s] LLM CALL (iteration 1)
[+1.38s]   -> LLM responded with stop_reason='tool_use' (latency: 1375ms)
[+1.38s] TOOL CALL: get_rewrite_suggestion
[+31.42s] ERROR: HTTPSConnectionPool(host='teachmewrite-ai-195673983724.asia-south1.run.app', port=443): Read timed out. (read timeout=30)
[+31.42s]   -> Tool result: ERROR (latency: 30039ms)
[+31.42s] LLM CALL (iteration 2)
[+32.93s]   -> LLM responded with stop_reason='end_turn' (latency: 1515ms)
[+32.93s] FINAL ANSWER (336 chars)
[+244.52s] USER: I am agree with my father that education is more better than money. — Just fix i
[+244.52s] LLM CALL (iteration 1)
[+245.70s]   -> LLM responded with stop_reason='tool_use' (latency: 1182ms)
[+245.70s] TOOL CALL: get_rewrite_suggestion
[+250.69s]   -> Tool result: OK (latency: 4990ms)
[+250.69s] LLM CALL (iteration 2)
[+251.78s]   -> L

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=8),
    retry=retry_if_exception_type((requests.exceptions.RequestException,)),
    reraise=True
)
def _call_feedback_resilient(student_text: str, level: int) -> dict:
    r = requests.post(f"{LIVE_URL}/feedback",
                      json={"student_text": student_text, "level": level}, timeout=60)
    r.raise_for_status()
    return r.json()

def get_writing_feedback_v2(student_text: str, level: int) -> dict:
    return _call_feedback_resilient(student_text, level)

def get_rewrite_suggestion_v2(student_text: str, level: int) -> dict:
    result = _call_feedback_resilient(student_text, level)
    return {"rewrite_suggestion": result.get("rewrite_suggestion", "")}

# Update the tool dispatcher
TOOL_FUNCTIONS["get_writing_feedback"] = get_writing_feedback_v2
TOOL_FUNCTIONS["get_rewrite_suggestion"] = get_rewrite_suggestion_v2

print("✅ Cell complete. Tools now have retry + 60s timeout.")

✅ Cell complete. Tools now have retry + 60s timeout.


In [ ]:
print("=" * 60)
print("VERIFICATION RUN (with retry + 60s timeout)")
print("=" * 60)

# New agent instance (uses patched TOOL_FUNCTIONS)
agent_v2 = TracedAgent()
result_v2 = agent_v2.chat(query, level=5)

print("\n--- RESPONSE ---")
print(result_v2["response"])
result_v2["tracer"].print_timeline()

assert len([e for e in result_v2["tracer"].events if e["type"] == "error"]) == 0, "Should have no errors"
print("✅ Verification run passed. No errors.")


VERIFICATION RUN (with retry + 60s timeout)

--- RESPONSE ---
Here's the corrected version:

**"I agree with my father that education is better than money."**

Good work! That's a strong statement. 👍

AGENT TRACE TIMELINE
[+0.00s] USER: I am agree with my father that education is more better than money. — Just fix i
[+0.00s] LLM CALL (iteration 1)
[+1.24s]   -> LLM responded with stop_reason='tool_use' (latency: 1238ms)
[+1.24s] TOOL CALL: get_rewrite_suggestion
[+5.81s]   -> Tool result: OK (latency: 4573ms)
[+5.81s] LLM CALL (iteration 2)
[+6.62s]   -> LLM responded with stop_reason='end_turn' (latency: 802ms)
[+6.62s] FINAL ANSWER (137 chars)
Total elapsed: 6.62s
✅ Verification run passed. No errors.
